# 10-Mode Step-Index Multimode Soliton, Cached CP Pathway

Production run notebook for the 1550 nm step-index soliton benchmark using the cached CP RHS and FFTW-threaded Raman convolution. This version is wired to the 2026-06-17 collaborator data drop and defaults to the 10-mode comparison case.


In [ ]:
using Pkg

const ROOT = abspath(joinpath(@__DIR__, ".."))
Pkg.activate(ROOT)

using FFTW
using LinearAlgebra
using Printf
using Random
using Statistics
using PyPlot

include(joinpath(ROOT, "src", "PulsePropagation.jl"))
using .PulsePropagation


In [ ]:

MODE_COUNTS = [5, 10, 20, 30, 40]
RUN_MODES = [10]

BENCHMARK_TAG = "5e-05"
BENCHMARK_DATA_DIR = joinpath(@__DIR__, "benchmark_data", "20260617")

L_FIBER_M = 10.0
DZ_M = 1e-3;#5e-5
SAVE_EVERY_STEPS = 100
SAVE_Z = collect(0:SAVE_EVERY_STEPS:round(Int, L_FIBER_M / DZ_M)) .* DZ_M
@assert SAVE_Z[1] == 0.0
@assert isapprox(SAVE_Z[end], L_FIBER_M; atol=1e-12)

NT = 2^13
TIME_WINDOW_PS = 100.0
PULSE_FWHM_PS = 0.5
TOTAL_ENERGY_NJ = 200.0
ENERGY_PER_LAUNCHED_MODE_NJ = 20.0
TIME_OFFSET_PS = -30.0
RECONSTRUCTED_INPUT_CENTER_PS = -30.0024
RECONSTRUCTED_INPUT_SIGMA_PS = 0.150125
LAMBDA0_M = 1550e-9
N2_M2_PER_W = 2.3e-20
RAMAN_FRACTION = 0.245

TENSOR_ERROR = 1e-2
CP_TARGET_ERROR = TENSOR_ERROR
CP_INITIAL_RANK = 8
CP_RANK_GROWTH = 2
CP_MAXITER = 260
CP_TOL = 1e-9
CP_CHECK_EVERY = 10
CP_SEED = 11
REFINED_RANK_RESOLUTION = 4

FFTW_THREADS = 8
BLAS_THREADS = 4
FFTW_FLAGS = :MEASURE
USE_REFERENCE_INPUT_FIELDS = true
REFERENCE_FIELD_BASIS = :reference
SOLVER_FIELD_BASIS = :solver

# PyTorch/collaborator and solver/reference mode profiles differ by signs for these real modes.
# S_torch[i,j,k,l] = prod(MODE_SIGNS_SOLVER_TO_REFERENCE[[i,j,k,l]]) * S_solver[i,j,k,l].
MODE_SIGNS_SOLVER_TO_REFERENCE = [
    1, 1, -1, 1, 1, 1, 1, -1, -1, 1,
    1, 1, 1, 1, 1, -1, 1, -1, -1, -1,
    1, -1, 1, -1, 1, 1, 1, 1, 1, 1,
    -1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
]

max_rank_for_modes(num_modes) = num_modes <= 10 ? 256 : 1024

@printf("Nt = %d, L = %.1f m, dz = %.4g m, save spacing = %.3g m, saved z points = %d\n",
        NT, L_FIBER_M, DZ_M, SAVE_EVERY_STEPS * DZ_M, length(SAVE_Z))
@printf("Benchmark data dir: %s\n", BENCHMARK_DATA_DIR)


In [ ]:

function _parse_npy_header(header::AbstractString)
    descr_match = match(r"'descr'\s*:\s*'([^']+)'", header)
    fortran_match = match(r"'fortran_order'\s*:\s*(True|False)", header)
    shape_match = match(r"'shape'\s*:\s*\(([^\)]*)\)", header)
    descr_match === nothing && error("Could not parse NPY dtype from header: $header")
    fortran_match === nothing && error("Could not parse NPY order from header: $header")
    shape_match === nothing && error("Could not parse NPY shape from header: $header")
    shape = Tuple(parse(Int, strip(s)) for s in split(shape_match.captures[1], ',') if !isempty(strip(s)))
    return descr_match.captures[1], fortran_match.captures[1] == "True", shape
end

function _npy_eltype(descr::AbstractString)
    descr in ("<c16", "|c16") && return ComplexF64
    descr in ("<c8", "|c8") && return ComplexF32
    descr in ("<f8", "|f8") && return Float64
    descr in ("<f4", "|f4") && return Float32
    error("Unsupported NPY dtype $descr")
end

function read_npy(path::AbstractString)
    open(path, "r") do io
        magic = read(io, 6)
        magic == UInt8[0x93, 0x4e, 0x55, 0x4d, 0x50, 0x59] || error("$path is not an NPY file")
        major = read(io, UInt8)
        _minor = read(io, UInt8)
        header_len = major == 1 ? Int(ltoh(read(io, UInt16))) : Int(ltoh(read(io, UInt32)))
        descr, fortran_order, shape = _parse_npy_header(String(read(io, header_len)))
        T = _npy_eltype(descr)
        data = Vector{T}(undef, prod(shape))
        read!(io, data)
        if fortran_order || length(shape) <= 1
            return reshape(data, shape...)
        end
        arr = reshape(data, reverse(shape)...)
        return permutedims(arr, reverse(1:length(shape)))
    end
end

benchmark_path(kind::AbstractString, num_modes) =
    joinpath(BENCHMARK_DATA_DIR, @sprintf("%s_%d_%s.npy", kind, num_modes, BENCHMARK_TAG))

has_benchmark_array(kind::AbstractString, num_modes) = isfile(benchmark_path(kind, num_modes))

load_collaborator_coefficients(num_modes) = vec(ComplexF64.(read_npy(benchmark_path("coeffs", num_modes))))

function load_collaborator_fields(kind::AbstractString, num_modes)
    raw = ComplexF64.(read_npy(benchmark_path(kind, num_modes)))
    size(raw) == (num_modes, NT) || error("Expected $(kind) array with shape ($(num_modes), $(NT)), got $(size(raw))")
    fields = Array{ComplexF64}(undef, NT, num_modes, 1)
    fields[:, :, 1] .= permutedims(raw, (2, 1))
    return fields
end

mode_signs(num_modes) = Float64.(MODE_SIGNS_SOLVER_TO_REFERENCE[1:num_modes])

function apply_mode_signs(fields, num_modes)
    signs = reshape(mode_signs(num_modes), 1, num_modes, 1)
    return ComplexF64.(fields .* signs)
end

function reference_to_solver_basis(fields, num_modes)
    SOLVER_FIELD_BASIS == :solver || error("Only SOLVER_FIELD_BASIS=:solver is currently supported.")
    REFERENCE_FIELD_BASIS == :solver && return ComplexF64.(fields)
    REFERENCE_FIELD_BASIS == :reference && return apply_mode_signs(fields, num_modes)
    error("Unsupported REFERENCE_FIELD_BASIS=$(REFERENCE_FIELD_BASIS)")
end

function solver_to_reference_basis(fields, num_modes)
    SOLVER_FIELD_BASIS == :solver || error("Only SOLVER_FIELD_BASIS=:solver is currently supported.")
    REFERENCE_FIELD_BASIS == :solver && return ComplexF64.(fields)
    REFERENCE_FIELD_BASIS == :reference && return apply_mode_signs(fields, num_modes)
    error("Unsupported REFERENCE_FIELD_BASIS=$(REFERENCE_FIELD_BASIS)")
end

function reconstructed_collaborator_input_fields(num_modes)
    grid = TimeGrid(NT, TIME_WINDOW_PS)
    t = time_axis(grid)
    coeffs = load_collaborator_coefficients(num_modes)
    shape = exp.(-((t .- RECONSTRUCTED_INPUT_CENTER_PS).^2) ./ (2RECONSTRUCTED_INPUT_SIGMA_PS^2))
    amp = sqrt(ENERGY_PER_LAUNCHED_MODE_NJ * 1e3 / (sum(abs2, shape) * grid.dt))
    profile = amp .* shape
    fields = zeros(ComplexF64, NT, num_modes, 1)
    for m in 1:num_modes
        fields[:, m, 1] .= coeffs[m] .* profile
    end
    return fields
end

function reference_or_reconstructed_fields(num_modes, dofs)
    fields = if USE_REFERENCE_INPUT_FIELDS && has_benchmark_array("input_fields", num_modes)
        load_collaborator_fields("input_fields", num_modes)
    else
        reconstructed_collaborator_input_fields(num_modes)
    end
    return reference_to_solver_basis(fields, num_modes)
end

relative_field_error(a, b) = norm(vec(a .- b)) / norm(vec(b))

function compare_input_to_reference(problem, num_modes)
    has_benchmark_array("input_fields", num_modes) || return nothing
    reference = load_collaborator_fields("input_fields", num_modes)
    problem_fields = solver_to_reference_basis(problem.initial_state.fields, num_modes)
    return (; relerr=relative_field_error(problem_fields, reference),
            maxabs=maximum(abs.(problem_fields .- reference)))
end

function final_fields_in_reference_basis(sol, num_modes)
    return solver_to_reference_basis(sol.output.fields[:, :, end:end], num_modes)
end

function compare_final_to_reference(sol, num_modes)
    has_benchmark_array("output_fields", num_modes) || return nothing
    reference = load_collaborator_fields("output_fields", num_modes)
    final_fields = final_fields_in_reference_basis(sol, num_modes)
    return (; relerr=relative_field_error(final_fields, reference),
            maxabs=maximum(abs.(final_fields .- reference)))
end

function cp_relative_error(cp, tensor, tensor_norm=norm(tensor))
    return norm(PulsePropagation.cp_reconstruct(cp, tensor) .- tensor) / tensor_norm
end

function fit_cp_rank(tensor, rank; init, rng, compression, tensor_norm)
    elapsed = @elapsed cp, history = PulsePropagation.cp_als_warm(
        tensor, rank;
        init=init,
        maxiter=compression.maxiter,
        tol=compression.tol,
        ridge=compression.ridge,
        check_every=compression.check_every,
        rng=rng,
        verbose=compression.verbose,
    )
    err = cp_relative_error(cp, tensor, tensor_norm)
    return (; cp, history, elapsed, error=err)
end

function resolve_refined_cp_compression(compression::CPCompression, fiber, sim;
                                        rank_resolution=REFINED_RANK_RESOLUTION)
    compression.tensor !== nothing &&
        return PulsePropagation._resolve_cp_compression(compression, fiber, sim)
    sim.scalar || error("Refined CP compression currently supports scalar modal propagation.")

    tensor = Float64.(fiber.sr)
    tensor_norm = norm(tensor)
    rng = Random.MersenneTwister(compression.seed)
    attempts = NamedTuple[]

    lower = nothing
    upper = nothing
    init = nothing
    rank = compression.initial_rank

    while rank <= compression.max_rank
        fit = fit_cp_rank(tensor, rank; init, rng, compression, tensor_norm)
        push!(attempts, (; phase=:coarse, rank, error=fit.error,
                         elapsed=fit.elapsed, history=fit.history))
        @printf("coarse rank %4d: tensor error = %.3e, fit %.2f s\n",
                rank, fit.error, fit.elapsed)
        if fit.error <= compression.target_error
            upper = (; rank, cp=fit.cp, error=fit.error)
            break
        end
        lower = (; rank, cp=fit.cp, error=fit.error)
        init = fit.cp
        rank *= compression.rank_growth
    end

    upper === nothing && error("No CP rank met target_error=$(compression.target_error) up to max_rank=$(compression.max_rank).")
    if lower === nothing
        return (; tensor=upper.cp, rank=upper.rank, error=upper.error,
                target_error=compression.target_error, attempts, fitted=true,
                elapsed=sum(a.elapsed for a in attempts))
    end

    while upper.rank - lower.rank > 4rank_resolution
        candidate = lower.rank + cld(upper.rank - lower.rank, 2)
        fit = fit_cp_rank(tensor, candidate;
                          init=lower.cp, rng, compression, tensor_norm)
        push!(attempts, (; phase=:refine, rank=candidate, error=fit.error,
                         elapsed=fit.elapsed, history=fit.history))
        @printf("refine rank %4d: tensor error = %.3e, fit %.2f s\n",
                candidate, fit.error, fit.elapsed)
        if fit.error <= compression.target_error
            upper = (; rank=candidate, cp=fit.cp, error=fit.error)
        else
            lower = (; rank=candidate, cp=fit.cp, error=fit.error)
        end
    end

    sweep_start = lower.rank
    sweep_stop = upper.rank
    sweep_init = nothing
    for candidate in sweep_start:rank_resolution:sweep_stop
        fit = fit_cp_rank(tensor, candidate;
                          init=sweep_init, rng, compression, tensor_norm)
        push!(attempts, (; phase=:sweep, rank=candidate, error=fit.error,
                         elapsed=fit.elapsed, history=fit.history))
        @printf("sweep rank %4d: tensor error = %.3e, fit %.2f s\n",
                candidate, fit.error, fit.elapsed)
        if fit.error <= compression.target_error
            return (; tensor=fit.cp, rank=candidate, error=fit.error,
                    target_error=compression.target_error, attempts,
                    fitted=true, elapsed=sum(a.elapsed for a in attempts))
        end
        sweep_init = fit.cp
    end

    return (; tensor=upper.cp, rank=upper.rank, error=upper.error,
            target_error=compression.target_error, attempts, fitted=true,
            elapsed=sum(a.elapsed for a in attempts))
end


In [ ]:

function build_step_index_problem(num_modes; target_error=CP_TARGET_ERROR,
                                  max_rank=max_rank_for_modes(num_modes),
                                  fftw_threads=FFTW_THREADS,
                                  fftw_flags=FFTW_FLAGS,
                                  blas_threads=BLAS_THREADS)
    grid = TimeGrid(NT, TIME_WINDOW_PS)
    fiber_folder = joinpath(ROOT, "step_wavelength1550nm_10mode_csv")
    system, dofs = load_fiber_system(
        fiber_folder;
        modes=1:num_modes,
        length=L_FIBER_M,
        lambda0=LAMBDA0_M,
        material=Silica(),
        betas_filename="betas.csv",
        s_tensors_filename="S_tensors_10modes.csv",
                scalar=true,
        n2=N2_M2_PER_W,
        raman_fraction=RAMAN_FRACTION,
    )

    fields = reference_or_reconstructed_fields(num_modes, dofs)

    compression = CPCompression(
        target_error=target_error,
        initial_rank=CP_INITIAL_RANK,
        max_rank=max_rank,
        rank_growth=CP_RANK_GROWTH,
        maxiter=CP_MAXITER,
        tol=CP_TOL,
        check_every=CP_CHECK_EVERY,
        seed=CP_SEED,
        verbose=false,
    )

    solver = RK4IPSolver(
        stepping=FixedStep(dz=DZ_M),
        saveat=saveat = SaveAt([0.0, L_FIBER_M]),#SaveAt(SAVE_Z),
        compression=compression,
        cp_rhs=CachedCPRHS(fftw_threads=fftw_threads, fftw_flags=fftw_flags, blas_threads=blas_threads),
        reltol=1e-6,
        abstol=1e-6,
    )

    return PulsePropagationProblem(
        ; grid, system, dofs,
        terms=PropagationTerms(Dispersion(), Kerr(), Raman(:model)),
        solver, fields,
    )
end


In [ ]:

num_modes = RUN_MODES[1]
problem = build_step_index_problem(num_modes)
input_compare = compare_input_to_reference(problem, num_modes)
@show input_compare

objs = PulsePropagation.backend_objects(problem)

cpinfo = resolve_refined_cp_compression(
    problem.solver.compression,
    objs.fiber,
    objs.sim,
)

@printf("CP rank = %d, tensor error = %.3e\n", cpinfo.rank, cpinfo.error)
cpinfo.rank, cpinfo.error, cpinfo.tensor


In [ ]:

function backend_fiber_with_length(fiber, L)
    return PulsePropagation.Fiber{Float64}(
        ; betas=fiber.betas, sr=fiber.sr, L0=Float64(L), n2=fiber.n2,
        material=fiber.material, mfd=fiber.mfd, mm_folder=fiber.mm_folder,
        fr=fiber.fr,
    )
end

function backend_sim_with_dz(sim, dz)
    return PulsePropagation.Simulation{Float64}(
        ; lambda0=sim.lambda0, f0=sim.f0, dz=Float64(dz), save_period=0.0,
        midx=sim.midx, scalar=sim.scalar, ellipticity=sim.ellipticity,
        include_Raman=sim.include_Raman, gain_model=sim.gain_model,
        pulse_centering=sim.pulse_centering, progress_bar=false,
        step_method=sim.step_method, cs=sim.cs,
        cs_model=sim.cs_model, betas=sim.betas, source=sim.source,
    )
end

function one_step_cp_outputs(objs, cp_tensor, dz; fftw_threads=FFTW_THREADS, fftw_flags=FFTW_FLAGS)
    fiber = backend_fiber_with_length(objs.fiber, dz)
    sim = backend_sim_with_dz(objs.sim, dz)
    zsave = [0.0, dz]
    rk4 = PulsePropagation.propagate_rk4ip_cp_cached(
        fiber, objs.ic, sim, cp_tensor;
        zsave, dz, fftw_threads, fftw_flags=PulsePropagation._fftw_flags(fftw_flags),
        blas_threads=BLAS_THREADS,
    )
    tsit = PulsePropagation.propagate_ode_cp_cached(
        fiber, objs.ic, sim, cp_tensor;
        zsave, fixed_dt=dz, reltol=1e-12, abstol=1e-12,
        fftw_threads, fftw_flags=PulsePropagation._fftw_flags(fftw_flags),
        blas_threads=BLAS_THREADS,
    )
    ref = PulsePropagation.propagate_ode_cp_cached(
        fiber, objs.ic, sim, cp_tensor;
        zsave, reltol=1e-12, abstol=1e-12, alg=PulsePropagation.DifferentialEquations.Vern9(),
        fftw_threads, fftw_flags=PulsePropagation._fftw_flags(fftw_flags),
        blas_threads=BLAS_THREADS,
    )
    return rk4.fields[:, :, end], tsit.fields[:, :, end], ref.fields[:, :, end]
end

# first_step_relerr(a, b) = norm(vec(a .- b)) / norm(vec(b))
# first_step_update_relerr(a, b, a0, dz) =
#     norm(vec(((a .- a0) ./ dz) .- ((b .- a0) ./ dz))) / norm(vec((b .- a0) ./ dz))

# initial_fields = objs.ic.fields[:, :, end]
# first_step_dzs = [2e-3, 1e-3, 5e-4, 2.5e-4, 1e-4, 5e-5, 1e-5]
# first_step_errors = NamedTuple[]

# @printf("%10s %14s %14s %14s %14s\n", "dz", "RK4/ref", "Tsit/ref", "RK4/Tsit", "upd RK4/Tsit")
# for dz in first_step_dzs
#     rk4_1, tsit_1, ref_1 = one_step_cp_outputs(objs, cpinfo.tensor, dz)
#     row = (;
#         dz,
#         rk4_ref=first_step_relerr(rk4_1, ref_1),
#         tsit_ref=first_step_relerr(tsit_1, ref_1),
#         rk4_tsit=first_step_relerr(rk4_1, tsit_1),
#         update_rk4_tsit=first_step_update_relerr(rk4_1, tsit_1, initial_fields, dz),
#     )
#     push!(first_step_errors, row)
#     @printf("%10.2e %14.6e %14.6e %14.6e %14.6e\n",
#             row.dz, row.rk4_ref, row.tsit_ref, row.rk4_tsit, row.update_rk4_tsit)
# end

# first_step_errors


In [ ]:

solver_with_tensor = RK4IPSolver(
    stepping=problem.solver.stepping,
    saveat=problem.solver.saveat,
    compression=CPCompression(; tensor=cpinfo.tensor, target_error=cpinfo.error),
    cp_rhs=problem.solver.cp_rhs,
    reltol=problem.solver.reltol,
    abstol=problem.solver.abstol,
)

problem_with_tensor = PulsePropagationProblem(
    initial_state=problem.initial_state,
    model=problem.model,
    solver=solver_with_tensor,
)

propagation_time_s = @elapsed sol = solve(problem_with_tensor)
final_compare = compare_final_to_reference(sol, num_modes)

@printf("Propagation time = %.3f s\n", propagation_time_s)
@show final_compare


In [ ]:
# # Optional fixed-step fourth-order RK4IP comparison through the same compressed CP pathway.
# # The first-step study suggested RK4IP needs about half the Tsit5 step size.
# RK4_DZ_M = DZ_M / 2
# RK4_BLAS_THREADS = BLAS_THREADS
# rk4_expected_steps = round(Int, L_FIBER_M / RK4_DZ_M)
# rk4_expected_rhs_evals = 4 * rk4_expected_steps

# @printf("RK4IP dz = %.4g m, expected steps = %d, expected nonlinear RHS evals = %d
# ",
#         RK4_DZ_M, rk4_expected_steps, rk4_expected_rhs_evals)

# rk4_solver_with_tensor = RK4IPSolver(
#     stepping=FixedStep(dz=RK4_DZ_M),
#     saveat=problem.solver.saveat,
#     compression=CPCompression(; tensor=cpinfo.tensor, target_error=cpinfo.error),
#     cp_rhs=FixedRK4IPCPRHS(
#         fftw_threads=FFTW_THREADS,
#         fftw_flags=FFTW_FLAGS,
#         blas_threads=RK4_BLAS_THREADS,
#     ),
#     reltol=problem.solver.reltol,
#     abstol=problem.solver.abstol,
# )a

# rk4_problem_with_tensor = PulsePropagationProblem(
#     initial_state=problem.initial_state,
#     model=problem.model,
#     solver=rk4_solver_with_tensor,
# )

# # Uncomment when you want the full fixed-step RK4IP comparison.
# # rk4_propagation_time_s = @elapsed rk4_sol = solve(rk4_problem_with_tensor)
# # rk4_final_compare = compare_final_to_reference(rk4_sol, num_modes)
# # @printf("RK4IP propagation time = %.3f s
# ", rk4_propagation_time_s)
# # @show rk4_final_compare


In [ ]:

stats = sol.output.ode_sol.stats

@printf("RHS evals      = %d\n", stats.nf)
@printf("Accepted steps = %d\n", stats.naccept)
@printf("Rejected steps = %d\n", stats.nreject)


In [ ]:

if has_benchmark_array("output_fields", num_modes)
    ref_final = load_collaborator_fields("output_fields", num_modes)[:, :, 1]
    julia_final = final_fields_in_reference_basis(sol, num_modes)[:, :, 1]
    mode_to_plot = min(num_modes, 1)
    figure(figsize=(6,4))
    plot(abs2.(julia_final[:, mode_to_plot]), label="Julia CP")
    plot(abs2.(ref_final[:, mode_to_plot]), "--", label="Collaborator")
    yscale("log")
    #xlim(2000,3000)
    ylim(1e-8, nothing)
    title(@sprintf("Mode %d final temporal power", mode_to_plot))
    legend()
    display(gcf())
end


In [ ]:

if has_benchmark_array("output_fields", num_modes)
    ref_final = load_collaborator_fields("output_fields", num_modes)[:, :, 1]
    julia_final = final_fields_in_reference_basis(sol, num_modes)[:, :, 1]
    mode_to_plot = min(num_modes, 10)
    figure(figsize=(6,4))
    plot(fftshift(abs2.(ifft(julia_final[:, mode_to_plot], 1))), label="Julia CP")
    plot(fftshift(abs2.(ifft(ref_final[:, mode_to_plot], 1))), "--", label="Collaborator")
    yscale("log")
    ylim(1e-8, nothing)
    title(@sprintf("Mode %d final spectrum", mode_to_plot))
    legend()
    display(gcf())
end


In [ ]:

figure(figsize=(6,4))
plot(abs2.(ref_final[:, :]), "--", label="Collaborator")
plot(abs2.(julia_final[:, :]), label="Julia CP", color="k", "--", alpha = 0.5)
yscale("log")
xlim(1500,7000)
ylim(1e-2, nothing)
#title(@sprintf("Mode %d final temporal power", mode_to_plot))
#legend()
display(gcf())



In [ ]:
# Optional fixed-step fourth-order RK4IP comparison through the same compressed CP pathway.
# The first-step study suggested RK4IP needs about half the Tsit5 step size.
RK4_DZ_M = DZ_M / 2
RK4_BLAS_THREADS = BLAS_THREADS
rk4_expected_steps = round(Int, L_FIBER_M / RK4_DZ_M)
rk4_expected_rhs_evals = 4 * rk4_expected_steps

@printf("RK4IP dz = %.4g m, expected steps = %d, expected nonlinear RHS evals = %d
",
        RK4_DZ_M, rk4_expected_steps, rk4_expected_rhs_evals)

rk4_solver_with_tensor = RK4IPSolver(
    stepping=FixedStep(dz=RK4_DZ_M),
    saveat=problem.solver.saveat,
    compression=CPCompression(; tensor=cpinfo.tensor, target_error=cpinfo.error),
    cp_rhs=FixedRK4IPCPRHS(
        fftw_threads=FFTW_THREADS,
        fftw_flags=FFTW_FLAGS,
        blas_threads=RK4_BLAS_THREADS,
    ),
    reltol=problem.solver.reltol,
    abstol=problem.solver.abstol,
)

rk4_problem_with_tensor = PulsePropagationProblem(
    initial_state=problem.initial_state,
    model=problem.model,
    solver=rk4_solver_with_tensor,
)

# Uncomment when you want the full fixed-step RK4IP comparison.
rk4_propagation_time_s = @elapsed rk4_sol = solve(rk4_problem_with_tensor)
rk4_final_compare = compare_final_to_reference(rk4_sol, num_modes)
@printf("RK4IP propagation time = %.3f s", rk4_propagation_time_s)
@show rk4_final_compare


In [ ]:

if has_benchmark_array("output_fields", num_modes)
    ref_final = load_collaborator_fields("output_fields", num_modes)[:, :, 1]
    julia_final = final_fields_in_reference_basis(rk4_sol, num_modes)[:, :, 1]
    mode_to_plot = min(num_modes, 1)
    figure(figsize=(6,4))
    plot(abs2.(julia_final[:, mode_to_plot]), label="Julia CP")
    plot(abs2.(ref_final[:, mode_to_plot]), "--", label="Collaborator")
    yscale("log")
    #xlim(2000,3000)
    ylim(1e-8, nothing)
    title(@sprintf("Mode %d final temporal power", mode_to_plot))
    legend()
    display(gcf())
end


In [ ]:

if has_benchmark_array("output_fields", num_modes)
    ref_final = load_collaborator_fields("output_fields", num_modes)[:, :, 1]
    julia_final = final_fields_in_reference_basis(rk4_sol, num_modes)[:, :, 1]
    mode_to_plot = min(num_modes, 10)
    figure(figsize=(6,4))
    plot(fftshift(abs2.(ifft(julia_final[:, mode_to_plot], 1))), label="Julia CP")
    plot(fftshift(abs2.(ifft(ref_final[:, mode_to_plot], 1))), "--", label="Collaborator")
    yscale("log")
    ylim(1e-8, nothing)
    title(@sprintf("Mode %d final spectrum", mode_to_plot))
    legend()
    display(gcf())
end


In [ ]:

figure(figsize=(6,4))
plot(abs2.(ref_final[:, :]), "--", label="Collaborator")
plot(abs2.(julia_final[:, :]), label="Julia CP", color="k", "--", alpha = 0.5)
yscale("log")
xlim(1500,7000)
ylim(1e-2, nothing)
#title(@sprintf("Mode %d final temporal power", mode_to_plot))
#legend()
display(gcf())

